In [1]:
import xarray as xr
import numpy as np

In [2]:
gebco = xr.open_zarr("./resources/GEBCO/GEBCO_regridded.zarr", consolidated=True)
phy = xr.open_zarr("./resources/copernicus_marine_service/regridded/obs-mob_glo_phy_regridded.zarr", consolidated=True)

In [3]:
target_depth = -gebco.elevation.where(gebco.elevation < 0, 0)
bottom_var = phy.sel(depth=target_depth, method='nearest')
bottom_var = bottom_var.drop_vars('depth')

In [4]:
# 1. Sort depth to ensure ffill works correctly (surface to bottom)
# 2. Forward fill: this replaces NaNs below the seafloor with the last valid data
target_depth = -gebco.elevation.where(gebco.elevation < 0, 0)
phy_filled = phy.sortby('depth').ffill(dim='depth')

# 3. Now .sel(method='nearest') will work because there are no NaNs "below" the seabed
bottom_var = phy_filled.sel(depth=target_depth, method='nearest')

# 4. Clean up the extra coord
bottom_var = bottom_var.drop_vars('depth')

In [5]:
temp_bottom=bottom_var["to"]
temp_bottom.to_netcdf("./data/processed/dynamic/temp_bottom.nc")